In [ ]:
import sys
import numpy as np
import torch

sys.path.append("../")
from datasets.dataloader import get_wireless_dataloader

def angles_to_quaternion(azimuth, elevation):
    """
    Compute the quaternion [w, x, y, z] for a rotation Rz(azimuth)*Ry(elevation).
    - azimuth = rotation about global z-axis (yaw)
    - elevation = rotation about the new y-axis (pitch)
    Angles must be in radians.
    """
    w = np.cos(azimuth / 2.0) * np.cos(elevation / 2.0)
    x = -np.sin(azimuth / 2.0) * np.sin(elevation / 2.0)
    y =  np.cos(azimuth / 2.0) * np.sin(elevation / 2.0)
    z =  np.sin(azimuth / 2.0) * np.cos(elevation / 2.0)

    return np.array([w, x, y, z], dtype=np.float32)

dataloader = get_wireless_dataloader(
    "../datasets/outputs/conf_16x2_414u_5.0ghz_sbrRT_sc104.mat",
    batch_size=16,
    num_pc=16378,
    drop_last=True
)

receiver_data = []

for batch_idx, batch in enumerate(dataloader):
    # point_cloud might be shape [B, N, 3] or a list of length B
    point_clouds     = batch["point_cloud"]
    tx_positions     = batch["tx_position"]      # [B, 3] or [16, 3] if batch=16
    rx_positions     = batch["rx_position"]      # [B, 3]
    channel_matrices = batch["channel_matrix"]   # [B, num_tx, num_rx, 2]
    aoa_list         = batch["aoa"]              # list of length B or shape [B, 2, num_paths]
    env_dims         = batch["env_dims"]         # [B, 3, 2] if same for all, else list

    print("Shapes from the dataloader (batch index = {}):".format(batch_idx))
    if isinstance(point_clouds, torch.Tensor):
        print("  point_clouds:", point_clouds.shape)
    else:
        print("  point_clouds is a list of length:", len(point_clouds))
        print("    first item shape:", point_clouds[0].shape)

    print("  tx_positions:", tx_positions.shape)
    print("  rx_positions:", rx_positions.shape)
    print("  channel_matrices:", channel_matrices.shape)
    print("  aoa_list length:", len(aoa_list))
    print("  env_dims:", env_dims.shape)

    # Now process each item in this batch
    batch_size = rx_positions.shape[0]
    for i in range(batch_size):
        rx_pos    = rx_positions[i].cpu().numpy()   # shape [3]
        current_aoa = aoa_list[i]                   # shape [2, num_paths]

        # We assume the last column of AoA is the final bounce
        azimuth   = current_aoa[0, -1].item()  # float, in radians
        elevation = current_aoa[1, -1].item()  # float, in radians

        # Convert azimuth/elevation directly to quaternion
        quaternion = angles_to_quaternion(azimuth, elevation)

        receiver_data.append({
            "rx_position": rx_pos,
            "quaternion":  quaternion
        })

# Example: show some results
print("\nTotal receiver_data entries:", len(receiver_data))
for entry in receiver_data[:5]:
    print("Receiver Position:", entry["rx_position"],
          "Quaternion:", entry["quaternion"])


: 